# Dummy → Logistic Regression → Random Forest

Chưa chạy. Run All trên Colab với source + input của bước trước đã lưu bền vững. Không dùng dữ liệu trường/private. Đọc ml/COLAB_RUNBOOK.md.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile
REPO = Path("/content/student-advisor")
# Upload ONLY the source ZIP prepared for this project, never .env or private student data.
if not (REPO / "ml").exists():
    from google.colab import files
    uploaded = files.upload()
    source_zip = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    assert source_zip, "Upload student-advisor-colab-source.zip"
    with zipfile.ZipFile(source_zip) as archive:
        for member in archive.infolist():
            target = (REPO / member.filename).resolve()
            assert target.is_relative_to(REPO.resolve()), "Unsafe ZIP path"
            assert not (member.external_attr >> 16 & 0o170000) == 0o120000, "Symlink not allowed"
        archive.extractall(REPO)
assert (3, 12) <= sys.version_info[:2] < (3, 14), f"Use Python 3.12 or 3.13; current={sys.version.split()[0]}. Align/export environment with backend before final packaging."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO/"ml/requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(REPO/"packages/advisor_core")], check=True)
sys.path.insert(0, str(REPO))
# Optional: set True yourself and select your own project folder. No automatic Drive upload.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive/student-advisor-work") if USE_DRIVE else Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
print("WORK:", WORK, "Ephemeral runtime: download outputs before disconnecting." if not USE_DRIVE else "")
from ml.scripts.common import load, save, sha
from ml.scripts.bootstrap import record_environment
record_environment(REPO, WORK)
config = load(REPO/"ml/configs/experiment.json")


In [ ]:
# Read the commit only from the clean source archive manifest. Do not type a SHA here.
provenance=load(WORK/"environment.json")["source_provenance"]
assert provenance.get("source_kind")=="clean_git_commit_source_archive", "Rebuild a clean, committed source ZIP with Pack-ColabSource.ps1"
config["source_commit"] = provenance.get("source_commit")
config["source_provenance"] = provenance
SOURCE_COMMIT_APPROVED=False  # set True only after you verify the displayed commit in GitHub
config["source_commit_approved"] = SOURCE_COMMIT_APPROVED
import re
assert config["source_commit_approved"] and re.fullmatch("[a-f0-9]{40}", config["source_commit"] or ""), "Verify the archive commit in GitHub, then approve it"
from ml.scripts.train import train_and_select
selection = train_and_select(WORK/"processed", WORK/"runs/run-001", config)
print("Selected:", selection["selected"]["model"])
print("DEV only:", selection["selected"]["metrics"])
print("Beats dummy AP:", selection["beats_dummy_ap"])
